In [ ]:
# RAG 파이프라인 및 LangChain 관련 라이브러리
# langchain-text-splitters: 문서를 의미 있는 단위(청크)로 분할하는 도구
!pip install langchain-text-splitters==0.3.9
# tiktoken: OpenAI 모델이 텍스트를 처리하는 단위인 '토큰'을 계산하는 라이브러리
!pip install tiktoken==0.11.0
# langchain-community: 다양한 외부 도구(Vector Store, Loader 등)와 연동하는 커뮤니티 제공 모듈
!pip install langchain-community==0.3.27
# langchain-openai: OpenAI 모델을 LangChain에서 사용하기 위한 모듈
!pip install langchain-openai==0.3.31
# langchain-upstage: Upstage 모델을 LangChain에서 사용하기 위한 모듈
!pip install langchain-upstage==0.7.3

# Vector Store (벡터 데이터베이스)
# chromadb: 텍스트 임베딩(벡터)을 저장하고 검색하는 경량 벡터 DB
!pip install chromadb==1.0.20

# Document Loaders (다양한 형식의 문서 로드용)
# pypdf, pymupdf, pypdfium2: PDF 파일에서 텍스트를 추출하기 위한 라이브러리들
!pip install pypdf==4.3.1
!pip install pymupdf==1.26.3
!pip install pypdfium2==4.3.0

In [ ]:
# 구글 드라이브를 코랩 환경에 마운트.
# .env 파일과 같이 민감한 정보나 영구 저장할 파일(예: chroma_db)을 관리하기 위함.
from google.colab import drive

drive.mount('/content/drive')

# API 키 파일 및 데이터가 저장된 기본 경로를 설정.
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/09_RAG/'

In [ ]:
# .env 파일에서 환경 변수를 로드하기 위한 라이브러리.
from dotenv import load_dotenv
# 운영체제(Colab 런타임)의 환경 변수를 가져오기 위한 함수.
from os import getenv

# .env 파일을 로드하여 환경 변수를 설정.
# (경로가 올바르다면, 이전 노트북에서 생성한 .env 파일이 로드됨)
load_dotenv(base_path + '.env') # 로컬 환경 등에서는 이 경로를 활성화

# getenv 함수를 사용해 'UPSTAGE_API_KEY'라는 이름의 환경 변수 값을 가져옴.
UPSTAGE_API_KEY = getenv('UPSTAGE_API_KEY')

# API 키가 성공적으로 로드되었는지 확인하고 메시지를 출력.
if UPSTAGE_API_KEY:
    print('Success API Key Setting!')
else:
    # 키 로드 실패 시, 이전 노트북의 0-2 단계가 올바르게 실행되었는지 확인 필요
    print(f'ERROR: Failed to load UPSTAGE_API_KEY from {base_path}')

# 2. RAG

- RAG 파이프라인을 구축(이후 4-2 챕터에서 이를 도구로 사용하는 Agent로 확장될 예정)

## 2-1. RAG(Retrieval-Augmented Generation)란?

### 2-1-1. 검색-증강 생성

- LLM의 한계(잘못된 정보 생성-**환각**, 최신 정보 부족-**지식 컷오프**)를 극복하기 위한 기술.
- LLM이 '알고 있는' 내부 지식에만 의존하는 것이 아니라, '외부의 신뢰할 수 있는 데이터베이스'를 **검색(Retrieval)**하여, 이 정보를 **증강(Augmented)**한 프롬프트를 기반으로 답변을 **생성(Generation)**하는 방식.

![RAG](https://i.ibb.co/1tHKtKpj/Snipaste-2025-10-21-22-54-27.png)

1. **Retrieval (검색)**
    - 사용자의 질문이 들어오면, 이 질문과 관련된 문서를 LLM이 모르는 외부 지식 소스(Vector Store)에서 검색함.
    - (예: `shipping_polict.txt` 파일 내용)
2. **Augmented (증강)**
    - 검색된 문서(Context)를 사용자의 원본 질문과 함께 LLM에게 전달할 프롬프트에 포함시킴.
    - (예: "다음 <문서>를 참고하여 <질문>에 답해: <문서>... <질문>...")
3. **Generation (생성)**
    - LLM이 이 '증강된 프롬프트'를 바탕으로, '제공된 문서(Context)에 근거하여' 정확하고 신뢰할 수 있는 답변을 생성함.

### 2-1-2. Vector Store (벡터 스토어)

- RAG의 '외부 지식 소스' 역할을 하는, `수치형 벡터` 검색에 특화된 데이터베이스
1. **임베딩 (Embedding)**
    - RAG의 핵심 기술. 텍스트, 이미지 등 데이터를 고차원의 '벡터(숫자 배열)'로 변환하는 과정
    - '임베딩 모델'(LLM의 일종)이 이 변환을 수행함.
    - **핵심 특징:** 의미가 유사한 텍스트는 벡터 공간에서 '가까운 거리'에 위치하게 됨.
    - (예: "배송비"와 "shipping cost"는 매우 가까운 벡터로 변환됨)
2. **벡터 스토어의 역할**
    - **저장 (Ingestion):** 우리가 가진 문서(PDF, TXT)를 '청크'로 나누고, 각 청크를 '임베딩'하여 벡터로 변환한 뒤, (원본 청크, 벡터) 쌍으로 데이터베이스에 저장함
    - **검색 (Retrieval):**
        1. 사용자의 '질문' 역시 **동일한 임베딩 모델**을 사용해 '질문 벡터'로 변환.
        2. 벡터 스토어는 이 '질문 벡터'와 데이터베이스 내에 저장된 '문서 벡터'들 간의 유사도(거리)를 계산함 (예: 코사인 유사도, KNN).
        3. '질문 벡터'와 가장 가까운(유사한) 상위 K개의 '문서 벡터'를 찾고, 이에 해당하는 '원본 문서 청크'를 반환함.



---------


## 2-2. RAG 파이프라인 구축

1. **데이터 로드:** Vector Store에 저장할 원본 문서를 로드 (PDF, TXT 등).
2. **분할 (Chunking):** 로드된 문서를 `TextSplitter`로 잘게 분할.
3. **임베딩 및 저장 (Embedding & Storage):** `UpstageEmbeddings` 모델로 각 청크를 벡터화하고, `Chroma` Vector Store에 저장.
4. **리트리버 생성 (Retriever):** 저장된 Vector Store를 '검색'할 수 있는 `Retriever` 객체 생성.
5. **체인 구성 (Chain):** `Retriever`, `Prompt`, `LLM`, `Parser`를 LCEL(`|`)로 연결하여 RAG 체인 완성.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyMuPDFLoader  # PDF 로더
# glob: 파일 경로에서 와일드카드(*.pdf)를 사용해 패턴 매칭을 위한 라이브러리
import glob

# 1. 데이터 로드 (PDF)
# base_path 아래 /data/ 폴더에 있는 모든 .pdf 파일을 찾음
pdf_files = glob.glob(base_path + '/data/*.pdf')

# 모든 PDF 파일에서 로드된 문서(페이지)를 저장할 빈 리스트
documents = []
for pdf_filepath in pdf_files:
    # 각 PDF 파일을 PyMuPDFLoader를 사용하여 로드
    loader = PyMuPDFLoader(pdf_filepath)
    # .load(): PDF의 각 페이지를 별도의 Document 객체로 로드
    pages = loader.load()
    # 로드된 페이지들을 documents 리스트에 추가
    documents.extend(pages)

# 1. 데이터 로드 (TXT)
shipping_policy_file_path = base_path + '/shipping_policy.txt'
loader = TextLoader(shipping_policy_file_path)
text_documents = loader.load()

# 로드된 텍스트 문서(TXT)도 documents 리스트에 추가
documents.extend(text_documents)

print(f'총 문서(페이지+파일) 개수: {len(documents)}')
# 각 문서 일부 출력
for i, doc in enumerate(documents):
    print(f'\n문서 {i+1} 내용 일부:')
    print(doc.page_content[:100])  # 각 문서의 처음 100자만 출력

2. 데이터 청킹과 Vector Store 생성
    - 불러온 모든 문서(`documents` 리스트)를 청크로 분할
    - 분할된 청크를 임베딩하여 Chroma 데이터베이스에 저장

- **Chroma DataBase**
    - 벡터 스토어의 한 종류. 오픈 소스이며, 특히 소규모~중규모 프로젝트나 로컬 환경에서 테스트하기에 적합
    - 별도의 서버 설치 없이 Python 라이브러리처럼 작동하며, persist_directory를 지정하면 데이터를 디스크에 저장하여 영구적으로 사용할 수 있음
    - (대안: Pinecone(클라우드), Milvus(설치형), FAISS(라이브러리))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_upstage import UpstageEmbeddings

# 2. 분할 (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(documents)

# print(f'생성된 총 청크 개수: {len(chunks)}')

# 3. 임베딩 및 저장 (Embedding & Storage)

# 3-1. 임베딩 모델 초기화
# Upstage에서 제공하는 임베딩 모델(embedding-query)을 사용.
# (API 키는 환경 변수에서 자동 로드)
embeddings = UpstageEmbeddings(model='embedding-query')

# 3-2. 청크로부터 Chroma Vector Store 생성
# Chroma.from_documents()는 다음 작업을 한 번에 수행:
# 1. (Iterate) 'chunks' 리스트를 순회
# 2. (Embed) 각 청크의 텍스트를 'embeddings' 모델로 벡터화
# 3. (Store) (청크 텍스트, 메타데이터, 벡터)를 Chroma DB에 저장
vectorstore = Chroma.from_documents(
    documents=chunks,  # 저장할 청크 리스트
    embedding=embeddings,  # 벡터화를 위한 임베딩 모델
    persist_directory=base_path + 'chroma_db',  # DB를 저장할 디스크 경로
)

print('\n--- Vector Store 생성 완료 ---')
print(f"Vector Store에 저장된 문서 개수: {vectorstore._collection.count()}")

3. Retriver 생성
- `Vector Store`는 데이터베이스 자체를 의미함.
- `Retriever`는 이 데이터베이스에 '질문(query)'을 던져 '관련 문서(chunks)'를 가져오는 '검색기' 역할을 하는 LangChain의 추상화 인터페이스(Runnable).
- LCEL 체인에 연결(pipe)할 수 있도록 `vectorstore` 객체를 `retriever` 객체로 변환함.

In [ ]:
# 4. 리트리버 생성
# vectorstore 객체를 LangChain의 Retriever 인터페이스로 변환.
# .as_retriever()는 '질문(str)'을 입력받아 '문서 리스트(List[Document])'를 반환하는 Runnable이 됨.
retriever = vectorstore.as_retriever()

---------------

## 2-3. 단순한 RAG Chain 구현

- `retriever`와 `LCEL`을 연결해서 RAG 체인을 구현
- **RAG 체인의 데이터 흐름:**
    1. **입력:** `{'question': '...'}` (dict)
    2. `RunnablePassthrough.assign`: 입력 딕셔너리를 다음 단계로 통과시키면서, 새 키(`context`)를 계산하여 추가함
        - `itemgetter('question')`: 입력 딕셔너리에서 'question' 값(str)을 추출
        - `| retriever`: 추출된 질문(str)을 리트리버에 전달 -> `List[Document]` (검색된 청크들) 반환.
        - `| format_docs`: 검색된 문서 리스트를 하나의 긴 문자열(str)로 변환.
        - `assign(context=...)`: 계산된 문자열을 'context' 키에 할당. 
    3. **출력:** `{'question': '...', 'context': '...'}` (dict)
    4. `| rag_prompt`: 증강된 딕셔너리를 받아 프롬프트 템플릿에 주입 -> `PromptValue` 객체 생성
    5. `| model`: `PromptValue`를 LLM에 전달 -> `AIMessage` 객체 생성
    6. `| StrOutputParser`: `AIMessage`에서 텍스트 내용만 추출 -> 최종 답변(str) 반환

In [ ]:
# RunnablePassthrough: 입력을 수정 없이 그대로 다음 단계로 전달하는 역할.
# .assign()과 함께 쓰여, 원본 입력(question)을 유지하면서 검색 결과(context)를 추가할 때 유용.
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_upstage import ChatUpstage
# itemgetter: 딕셔너리에서 특정 키의 값을 추출하는 유틸리티 (lambda x: x['key']와 동일)
from operator import itemgetter

query = '주말에도 배송되나요?'

# 문서 리스트(List[Document])를 하나의 문자열로 포맷하는 함수
# (리트리버의 출력 형식을 프롬프트의 입력 형식에 맞추기 위함)
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# LLM 모델 정의
model = ChatUpstage()

# RAG용 프롬프트 템플릿 정의
# {context} (검색된 문서)와 {question} (원본 질문)을 변수로 받음
rag_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 친절한 고객 지원 담당자입니다.'),
        (
            'user',
            '다음 컨텍스트를 참고해서 질문에 답변해 주세요: **{context}**\n질문: {question}',
        ),
    ]
)

# 5. RAG 체인 구성 (LCEL 사용)
rag_chain = (
    # assign(): 'context' 키를 새로 생성하여 딕셔너리에 추가.
    # RunnablePassthrough가 원본 'question'은 그대로 통과시킴.
    RunnablePassthrough.assign(
        # 'context'의 값은 (질문 추출 -> 리트리버 검색 -> 포맷팅) 파이프라인으로 계산
        context=(itemgetter('question') | retriever | format_docs)
    )
    | rag_prompt  # {'question': ..., 'context': ...} 딕셔너리가 프롬프트로 전달됨
    | model  # 프롬프트가 모델로 전달됨
    | StrOutputParser()  # 모델의 출력(AIMessage)을 깔끔한 문자열(str)로 변환
)

# 체인 실행 (입력은 체인의 맨 처음 요구사항인 {'question': ...} 딕셔너리)
response = rag_chain.invoke({'question': query})

from pprint import pprint

print(f'\n질문: {query}')
pprint(f'답변: {response}')

-----------------

## 2-4. RAG 체인 정리 및 다음 단계: Agent

### 2-4-1. RAG 파이프라인 요약
- 이번 챕터에서는 **RAG (Retrieval-Augmented Generation)** 파이프라인의 전체 과정을 구축함.
- `Document Loaders` (`PyMuPDFLoader`, `TextLoader`)를 사용해 로컬의 PDF, TXT 파일을 불러옴. (`1. 로드`)
- `Text Splitter` (`RecursiveCharacterTextSplitter`)를 사용해 문서들을 검색에 용이한 작은 '청크' 단위로 분할함. (`2. 분할`)
- `Embeddings` (`UpstageEmbeddings`) 모델을 사용해 각 청크를 벡터(숫자 배열)로 변환함. (`3. 임베딩`)
- `Vector Store` (`Chroma`)에 임베딩된 벡터와 원본 청크를 저장함. (`4. 저장`)
- `vectorstore.as_retriever()`를 통해 질문 벡터와 가장 유사한 문서를 찾는 `Retriever` 객체를 생성함. (`5. 검색기 생성`)
- **LCEL (|)** 을 사용해 `Retriever`, `Prompt`, `LLM`, `Parser`를 연결(Chain)하여, 외부 문서에 근거한 답변을 생성하는 `rag_chain`을 완성함. (`6. 체인 구성`)


### 2-4-2. 단순 RAG 체인의 한계
- 우리가 만든 `rag_chain`은 **'문서 검색'** 이라는 한 가지 작업에 고정(Hard-coded)되어 있음.
- `rag_chain`은 사용자의 질문 의도와 상관없이 **무조건** `retriever`를 먼저 호출함.
- 만약 사용자가 **문서와 관련 없는 일반적인 질문**을 한다면 어떻게 될까?

```python
# 문서 내용(Yes24 배송 정책)과 관련 없는 질문
general_query = "오늘 날씨 어때?"

# RAG 체인은 이 질문조차도 벡터 스토어에서 검색하려고 시도함
response = rag_chain.invoke({"question": general_query})

print(f"질문: {general_query}")
print("---")
print(f"답변: {response}")

# 예상 출력:
# 질문: 오늘 날씨 어때?
# ---
# 답변: 죄송합니다. 제공된 컨텍스트(예스24 배송, 포인트 정책 등)에는 '오늘 날씨'에 대한 정보가 포함되어 있지 않습니다.
```


- 위와 같이, RAG 체인은 자신이 가진 문서(Context) 내에서만 답변을 시도하며, 일반 상식이나 실시간 정보(날씨, 뉴스 등)에 대해서는 답변할 수 없음.
- 만약 '배송 정책 질문'은 RAG로 답하고, '일반 상식 질문'은 LLM이 직접 답하게 하려면 어떻게 해야 할까?

### 2-4-3. [예고] 다음 단계: Agent와 Tools
- 이러한 한계를 극복하기 위해 **Agent** 개념이 도입됨.
- **Agent:** LLM을 '추론 엔진' 또는 '두뇌'로 사용하여, 사용자의 요청을 분석하고 **어떤 행동(Tool)을 취할지 스스로 결정**하게 만드는 방식.
- Tools: Agent가 사용할 수 있는 '도구'의 목록.
    - `Tool 1`: 우리가 만든 `rag_chain` (문서 검색 및 답변용)
    - `Tool 2`: `GoogleSearch` (실시간 정보 검색용)
    - `Tool 3`: `Calculator` (수학 계산용)
    - `Tool ...`: 기타 API 또는 사용자 정의 함수

**Agent의 작동 방식:**

1. 사용자가 "주말 배송 정책 알려줘"라고 질문함.
2. Agent(LLM)가 질문을 분석하고, 사용 가능한 도구 목록(`[rag_chain, GoogleSearch]`)을 살펴봄.
3. Agent가 "이 질문은 문서 검색이 필요하다. `rag_chain` 도구를 사용해야겠다"라고 **결정(Reasoning)**함.
4. Agent가 `rag_chain`을 호출하고 결과를 받음.
5. Agent가 `rag_chain`의 결과를 바탕으로 최종 답변을 생성함.

----

- 만약 사용자가 "오늘 날씨 어때?"라고 질문하면, Agent는 `rag_chain`이 아닌 `GoogleSearch` 도구를 사용하도록 결정할 것.
- 이처럼 Agent는 **'동적인 라우팅(Dynamic Routing)'**을 가능하게 하며, RAG 체인을 Agent가 사용할 수 있는 여러 도구 중 하나로 만들어 더 복잡하고 유연한 애플리케이션을 구축할 수 있게 함.
- 이후에는 이 `rag_chain`을 'Tool'로 변환하고, 이를 사용하는 Agent를 구축하는 방법을 알아볼 것.